# Monthly Order Summary
For each of the customer, produce the following summary per month

1. total orders
2. total items bought
3. total amount spend

In [0]:
silver_orders = spark.table("gizmobox.silver.orders")
silver_payments = spark.table("gizmobox.silver.payments")
silver_refunds = spark.table("hive_metastore.silver.refunds")

In [0]:
display(silver_orders.limit(1))
display(silver_payments.limit(1))
display(silver_refunds.limit(1))

In [0]:
from pyspark.sql import functions as F
silver_orders = (
    silver_orders
    .withColumn("order_month", F.date_format('transaction_timestamp', 'yyyy-MM'))
    .groupBy('order_month', 'customer_id')
    .agg(
        F.countDistinct('order_id').alias('total_orders'),
        F.sum('quantity').alias('total_items_bought'),
        F.sum(F.col('price') * F.col('quantity')).alias('total_amount')
    )
)
display(silver_orders)

In [0]:
silver_orders.writeTo("gizmobox.gold.py_order_monthly_summary").createOrReplace()

In [0]:
df = spark.table("gizmobox.gold.py_order_monthly_summary")
display(df)